In [ ]:
import sys, os
# Adjust depth based on notebook location relative to project root
_src_path = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']
well_meta = config.get('well_metadata', {})

print(f'Active experiment: {config.get("experiment_name", config.get("experiment_key", "?"))}')
print(f'Base dir: {base_dir}')

In [1]:
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
import phate
import math
import random
import gc
import scprep
from datetime import datetime, time
from matplotlib.animation import ImageMagickWriter
import matplotlib.animation as animation
import zipfile
from urllib.request import urlopen
import scipy.stats as st
from scipy.stats import norm
from scipy.stats import gaussian_kde
from scipy.stats import kde
from scipy.stats import binned_statistic
from scipy.stats import f_oneway
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['pdf.fonttype'] = 42
print(sns.__version__)
from anndata import AnnData
import scanpy as sc
from delve import *
import anndata as ad
from sklearn.preprocessing import MinMaxScaler
from kh import sketch
from sklearn.cluster import KMeans
import umap
print(sc.__version__)
today = datetime.now().strftime("%m%d%Y-%H%M")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
#Read back in the subsampled adata file
adata_save_path = os.path.join(base_dir, 'fullest_adata_nucArea_ge313_sub.h5ad')
fullest_adata_nucArea_ge313_sub = anndata.read_h5ad(adata_save_path)

In [ ]:
import numpy as np
from scipy import sparse

# --- Sanity check ---
A = fullest_adata_nucArea_ge313_sub
def has_bad(x):
    if sparse.issparse(x):
        data = x.data
        return {"nan_in_data": np.isnan(data).any(),
                "inf_in_data": np.isinf(data).any()}
    arr = np.asarray(x)
    return {"nan_any": np.isnan(arr).any(),
            "inf_any": np.isinf(arr).any()}
print("X shape:", A.X.shape)
print("Bad values in X:", has_bad(A.X))

# --- Cleaning ---
def clean_X(adata, clip_q=0.999):
    X = adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X)
    X = X.astype(np.float64, copy=True)
    X[~np.isfinite(X)] = np.nan
    med = np.nanmedian(X, axis=0)
    med[~np.isfinite(med)] = 0.0
    nan_rows, nan_cols = np.where(np.isnan(X))
    X[nan_rows, nan_cols] = med[nan_cols]
    hi = np.quantile(X, clip_q, axis=0)
    X = np.minimum(X, hi)
    adata.X = X.astype(np.float32)
    return adata

fullest_adata_nucArea_ge313_sub = clean_X(fullest_adata_nucArea_ge313_sub)
print("Any NaN left?", np.isnan(fullest_adata_nucArea_ge313_sub.X).any())


In [ ]:
# Adjusted list comprehension to exclude var_names containing "total"
columns_to_keep = [name for name in fullest_adata_nucArea_ge313_sub.var_names if "total_nuc_protein" not in name]

# Selecting the data with only the columns to keep
standard_trimmed_noPSTAT5_noTotal_adata_sub = fullest_adata_nucArea_ge313_sub[:, columns_to_keep]


In [ ]:
def laplacian_score_fs(adata = None,
                    k: int  = None,
                    n_jobs: int  = -1):

    X, feature_names, obs_names = parse_input(adata)
    W = construct_affinity(X = X, k = k, n_jobs = n_jobs)
    scores = laplacian_score(X = X, W = W)
    predicted_features = pd.DataFrame(scores, index = feature_names, columns = ['laplacian_score'])
    predicted_features = predicted_features.sort_values(by = 'laplacian_score', ascending = True)

    return predicted_features 

In [ ]:
n_obs = fullest_adata_nucArea_ge313_sub.n_obs
k_requested = 100
k_safe = max(1, min(k_requested, n_obs - 1))

l_score_standard = laplacian_score_fs(fullest_adata_nucArea_ge313_sub, k=k_safe)


In [ ]:
#l_score_fullest = laplacian_score_fs(adata_fullest_sub, k = 200)
l_score_standard = laplacian_score_fs(fullest_adata_nucArea_ge313_sub, k = 100)
#l_score_normalized = laplacian_score_fs(normalized_trimmed_noPSTAT5_adata_sub, k = 200)

In [ ]:
len(l_score_standard)

In [ ]:
l_score_standard.index[:46]

In [ ]:
# Extract relevant columns from adata_sub.obs
sample_ids = fullest_adata_nucArea_ge313_sub.obs['sample_ID']
treatment = fullest_adata_nucArea_ge313_sub.obs['treatment']

protein_list = (l_score_standard.index[:84])
for protein_name in protein_list:

    # Get the index of the protein in adata_sub.X
    protein_index = np.where(fullest_adata_nucArea_ge313_sub.var_names == protein_name)[0][0]

    # Specify the sample IDs you want to include
    selected_sample_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14]

    # Initialize the data arrays for violin plots and corresponding treatments
    data = []
    treatments = []

    # Generate data and corresponding treatments for each sample ID
    for sample_id in selected_sample_ids:
        # Filter data for the current sample ID
        sample_data = fullest_adata_nucArea_ge313_sub.X[sample_ids == sample_id, protein_index]

        # Filter treatment for the current sample ID
        sample_treatment = treatment[sample_ids == sample_id][0]

        # Check if sample_data is empty
        if len(sample_data) > 0:
            # Store the sample data and treatment
            data.append(sample_data)
            treatments.append(sample_treatment)

    # Create violin plots if there is data available
    if data:
        plt.figure(figsize=(12, 4))  # Set the size of the plot
        plt.violinplot(data, showmeans=False, showmedians=True, showextrema=False)  # Remove vertical lines

        # Add labels and legend
        plt.xlabel('Etoposide Exposure Duration')
        plt.ylabel('Z Normalized Intensity')
        plt.xticks(rotation=45)  # Rotate x-axis labels by 45 degrees
        plt.xticks(np.arange(1, len(selected_sample_ids) + 1), treatments)
        plt.title(protein_name)

        # Set custom y-axis limits
        plt.ylim(bottom=np.percentile(np.concatenate(data), 1), top=np.percentile(np.concatenate(data), 95))  # Adjust the percentile as needed

        # Show the plot
        plt.show()
